In [ ]:
import os
import numpy as np
import torchaudio
import matplotlib.pyplot as plt
from glob import glob
from collections import defaultdict
from speechbrain.pretrained import SepformerSeparation as separator

def batch_separate_and_evaluate(folder_path, output_dir, unlearn_speakers):
    all_metrics = defaultdict(list)
    flac_files = sorted(glob(os.path.join(folder_path, "*.flac")))

    if not flac_files:
        print("No .flac files found in the folder.")
        return

    for idx, file_path in enumerate(flac_files, 1):
        print(f"\n [{idx}/{len(flac_files)}] Processing: {file_path}")
        sub_output_dir = os.path.join(output_dir, f"output_{idx}")
        os.makedirs(sub_output_dir, exist_ok=True)

        model = separator.from_hparams(source="speechbrain/sepformer-libri3mix",
                                       savedir="pretrained_models/sepformer-librimix-3spk")
        mixture, sample_rate = torchaudio.load(file_path)
        if mixture.shape[0] > 1:
            mixture = mixture.mean(dim=0, keepdim=True)
        est_sources = model.separate_file(path=file_path)

        if est_sources.ndim != 3:
            raise ValueError(f"Unexpected shape for separated sources: {est_sources.shape}")

        num_speakers = est_sources.shape[2]
        for i in range(num_speakers):
            separated = est_sources[0, :, i].detach().cpu().unsqueeze(0)
            separated = match_length(mixture, separated)
            metrics = evaluate_metrics(mixture.squeeze(), separated.squeeze(), sample_rate)

            for key in metrics:
                all_metrics[key].append(metrics[key])

    # Compute average metrics
    avg_metrics = {key: round(np.mean(values), 2) for key, values in all_metrics.items()}
    print("\n Average Metrics Across All Files and Speakers:")
    for key, val in avg_metrics.items():
        print(f"{key}: {val}")

    # Plot and save as PNG
    plt.figure(figsize=(10, 5))
    metric_names = ["SDR", "SAR", "STOI", "MI", "SCI"]
    avg_values = [avg_metrics.get(metric, 0) for metric in metric_names]
    bars = plt.bar(metric_names, avg_values, color=["#009461","#ade812","#2596be"])
    for bar, value in zip(bars, avg_values):
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() - 0.5,
                 f'{value:.2f}', ha='center', va='top')
    plt.ylabel("Average Score")
    plt.title("Average Metrics Across LibriCSS Dataset")
    plt.grid(True, axis='y')
    plt.tight_layout()

    # Save plot
    plot_path = os.path.join(output_dir, "average_metrics.png")
    plt.savefig(plot_path)
    print(f"\n Metrics plot saved at: {plot_path}")

    plt.show()

if __name__ == "__main__":
    input_folder = input("Enter the folder path containing .flac files: ").strip()
    output_directory = input("Enter the output directory: ").strip()
    unlearn_input = input("Enter speaker numbers to unlearn (comma-separated, e.g., 1,3). Press Enter for none: ").strip()
    unlearn_speakers = list(map(int, unlearn_input.split(','))) if unlearn_input else []
    batch_separate_and_evaluate(input_folder, output_directory, unlearn_speakers)
